# cellmap-flow on Colab

Run cellmap-flow's inference server in Colab and serve it via a public ngrok
URL. Free Colab gives you a T4 GPU — ~10× faster than the CPU-only HF Space
tier. The URL only works while this Colab session is alive.

**Steps:**
1. Pick a runtime with GPU (Runtime → Change runtime type → T4 GPU).
2. Run all cells.
3. Copy the printed ngrok URL into the cellmap-flow browser dashboard's
   "Inference server URL" field.
4. Pan around in Neuroglancer; chunks are computed here on the Colab GPU.

## 1. Install cellmap-flow + ngrok

Takes ~3–5 minutes the first time.

In [ ]:
%pip install -q cellmap-flow huggingface_hub s3fs pyngrok

## 2. Configure model + dataset

Pick a cellmap HF model that ships ONNX or torch weights, and a public zarr
URL. Defaults below run a mito affinity model on a Janelia mouse-liver dataset.

In [ ]:
HF_REPO = "cellmap/jrc_mus-livers_16nm_to_8nm_mito"
MODEL_NAME = HF_REPO.split("/")[-1]
DATASET = "s3://janelia-cosem-datasets/jrc_mus-liver/jrc_mus-liver.zarr/recon-1/em/fibsem-uint8/"
PORT = 8765

## 3. Set up the public URL via ngrok

If you have an [ngrok account](https://dashboard.ngrok.com/get-started/your-authtoken),
paste your authtoken below for a stable URL and higher rate limits. Otherwise
you can leave it blank and ngrok will assign an anonymous URL.

In [ ]:
from pyngrok import ngrok, conf
import getpass

token = getpass.getpass("ngrok authtoken (leave empty for anonymous): ").strip()
if token:
    conf.get_default().auth_token = token

tunnel = ngrok.connect(PORT, "http")
PUBLIC_URL = tunnel.public_url
print("=" * 60)
print("Public URL (paste into the browser dashboard):")
print("  ", PUBLIC_URL)
print("=" * 60)

## 4. Start the cellmap-flow server

This blocks the cell. Keep this tab open while you use the URL above.

In [ ]:
!cellmap_flow_server huggingface --repo {HF_REPO} --name {MODEL_NAME} -d "{DATASET}" --port {PORT}